# Pneumonia Detection in Chest X-Rays

Deep-learning project for binary classification of chest X-rays into **NORMAL** and **PNEUMONIA** using a pretrained **ResNet50**.

The training strategy uses progressive fine-tuning in three stages:

1. Train only the final fully connected layer.
2. Train the final layer and `layer4`.
3. Fine-tune the complete network.

The notebook also evaluates the final model with a confusion matrix, classification metrics, ROC/AUC, and model interpretability using **Grad-CAM** and **Grad-CAM++**.


## 1. Imports and configuration

In [ ]:
from pathlib import Path
import copy
import random

import cv2
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import torch
import torch.nn as nn

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import train_test_split
from torch.optim import Adam
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, models, transforms
from tqdm.auto import tqdm

from pytorch_grad_cam import GradCAM, GradCAMPlusPlus
from pytorch_grad_cam.utils.image import show_cam_on_image


# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Paths
DATASET_ROOT = Path("../Datasets/Pneumonia Detection Dataset/chest_xray")
OUTPUT_DIR = Path("../outputs")
MODEL_DIR = Path("../models")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# Hyperparameters
IMG_SIZE = 224
BATCH_SIZE = 32
VAL_RATIO = 0.20
NUM_WORKERS = 0

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")


## 2. Dataset and preprocessing

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.9, 1.0)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])

eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])

train_root = DATASET_ROOT / "train"
test_root = DATASET_ROOT / "test"

# Two dataset objects point to the same training files but use different transforms.
train_dataset_full = datasets.ImageFolder(train_root, transform=train_transform)
val_dataset_full = datasets.ImageFolder(train_root, transform=eval_transform)
test_dataset = datasets.ImageFolder(test_root, transform=eval_transform)

targets = np.array(train_dataset_full.targets)
indices = np.arange(len(targets))

train_indices, val_indices = train_test_split(
    indices,
    test_size=VAL_RATIO,
    random_state=SEED,
    stratify=targets,
)

train_dataset = Subset(train_dataset_full, train_indices)
val_dataset = Subset(val_dataset_full, val_indices)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)

class_names = train_dataset_full.classes

print(f"Train size: {len(train_dataset)}")
print(f"Validation size: {len(val_dataset)}")
print(f"Test size: {len(test_dataset)}")
print(f"Classes: {class_names}")


## 3. ResNet50 model

In [ ]:
model = models.resnet50(
    weights=models.ResNet50_Weights.IMAGENET1K_V2
)

# Freeze the complete backbone initially.
for parameter in model.parameters():
    parameter.requires_grad = False

# Replace the original ImageNet classifier with a binary classifier.
num_features = model.fc.in_features
model.fc = nn.Linear(num_features, 2)

model = model.to(device)

criterion = nn.CrossEntropyLoss()

print(model.fc)


## 4. Training utilities

In [ ]:
def train_one_epoch(model, dataloader, optimizer, criterion, device):
    model.train()

    running_loss = 0.0
    running_correct = 0
    total = 0

    for images, labels in tqdm(dataloader, desc="Training", leave=False):
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad(set_to_none=True)

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        predictions = outputs.argmax(dim=1)

        running_loss += loss.item() * images.size(0)
        running_correct += (predictions == labels).sum().item()
        total += labels.size(0)

    return running_loss / total, running_correct / total


@torch.no_grad()
def evaluate_loss_accuracy(model, dataloader, criterion, device):
    model.eval()

    running_loss = 0.0
    running_correct = 0
    total = 0

    for images, labels in tqdm(dataloader, desc="Evaluation", leave=False):
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        predictions = outputs.argmax(dim=1)

        running_loss += loss.item() * images.size(0)
        running_correct += (predictions == labels).sum().item()
        total += labels.size(0)

    return running_loss / total, running_correct / total


In [ ]:
def configure_trainable_layers(model, phase):
    """Configure trainable ResNet50 layers for progressive fine-tuning."""

    for parameter in model.parameters():
        parameter.requires_grad = False

    if phase == 1:
        # Final classifier only
        for parameter in model.fc.parameters():
            parameter.requires_grad = True

    elif phase == 2:
        # Final classifier + last residual block
        for parameter in model.fc.parameters():
            parameter.requires_grad = True

        for parameter in model.layer4.parameters():
            parameter.requires_grad = True

    elif phase == 3:
        # Full fine-tuning
        for parameter in model.parameters():
            parameter.requires_grad = True

    else:
        raise ValueError("phase must be 1, 2, or 3")

    return [p for p in model.parameters() if p.requires_grad]


## 5. Progressive fine-tuning

In [ ]:
phases = [
    {
        "name": "Phase 1 - FC only",
        "phase": 1,
        "epochs": 5,
        "lr": 1e-3,
    },
    {
        "name": "Phase 2 - FC + layer4",
        "phase": 2,
        "epochs": 5,
        "lr": 1e-4,
    },
    {
        "name": "Phase 3 - Full fine-tuning",
        "phase": 3,
        "epochs": 10,
        "lr": 1e-5,
    },
]

best_val_loss = float("inf")
best_model_weights = copy.deepcopy(model.state_dict())
patience = 5

history = {
    "train_loss": [],
    "train_acc": [],
    "val_loss": [],
    "val_acc": [],
    "phase": [],
}

for config in phases:
    print(f"\n{'=' * 15} {config['name']} {'=' * 15}")

    trainable_parameters = configure_trainable_layers(
        model,
        config["phase"],
    )

    optimizer = Adam(
        trainable_parameters,
        lr=config["lr"],
    )

    # Patience is evaluated independently in each fine-tuning phase.
    patience_counter = 0

    for epoch in range(config["epochs"]):
        print(
            f"\nEpoch {epoch + 1}/{config['epochs']} "
            f"| LR={config['lr']:.0e}"
        )

        train_loss, train_acc = train_one_epoch(
            model,
            train_loader,
            optimizer,
            criterion,
            device,
        )

        val_loss, val_acc = evaluate_loss_accuracy(
            model,
            val_loader,
            criterion,
            device,
        )

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)
        history["phase"].append(config["name"])

        print(
            f"Train | Loss: {train_loss:.4f} | Accuracy: {train_acc:.4f}"
        )
        print(
            f"Val   | Loss: {val_loss:.4f} | Accuracy: {val_acc:.4f}"
        )

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_model_weights = copy.deepcopy(model.state_dict())
            patience_counter = 0
            print("New best model.")
        else:
            patience_counter += 1
            print(
                f"No validation-loss improvement "
                f"({patience_counter}/{patience})."
            )

        if patience_counter >= patience:
            print("Early stopping for this phase.")
            break

# Restore and save the best validation-loss checkpoint.
model.load_state_dict(best_model_weights)

model_path = MODEL_DIR / "best_resnet50_pneumonia.pth"
torch.save(model.state_dict(), model_path)

print(f"\nBest model saved to: {model_path}")
print(f"Best validation loss: {best_val_loss:.4f}")


## 6. Training history

In [ ]:
epochs = np.arange(1, len(history["train_loss"]) + 1)

plt.figure(figsize=(8, 5))
plt.plot(epochs, history["train_loss"], label="Train loss")
plt.plot(epochs, history["val_loss"], label="Validation loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training and Validation Loss")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(epochs, history["train_acc"], label="Train accuracy")
plt.plot(epochs, history["val_acc"], label="Validation accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Training and Validation Accuracy")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


## 7. Test-set evaluation

In [ ]:
test_loss, test_accuracy = evaluate_loss_accuracy(
    model,
    test_loader,
    criterion,
    device,
)

print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_accuracy:.4f}")


In [ ]:
@torch.no_grad()
def collect_predictions(model, dataloader, device):
    model.eval()

    y_true = []
    y_pred = []
    y_score = []

    for images, labels in tqdm(
        dataloader,
        desc="Collecting predictions",
        leave=False,
    ):
        images = images.to(device)

        logits = model(images)
        probabilities = torch.softmax(logits, dim=1)
        predictions = logits.argmax(dim=1)

        y_true.extend(labels.numpy())
        y_pred.extend(predictions.cpu().numpy())
        y_score.extend(probabilities[:, 1].cpu().numpy())

    return (
        np.asarray(y_true),
        np.asarray(y_pred),
        np.asarray(y_score),
    )


y_true, y_pred, y_score = collect_predictions(
    model,
    test_loader,
    device,
)

print(
    classification_report(
        y_true,
        y_pred,
        target_names=class_names,
        digits=4,
    )
)


### Confusion matrix

In [ ]:
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=class_names,
    yticklabels=class_names,
)

plt.xlabel("Predicted label")
plt.ylabel("True label")
plt.title("Confusion Matrix - Final Model")
plt.tight_layout()

confusion_matrix_path = OUTPUT_DIR / "confusion_matrix.png"
plt.savefig(confusion_matrix_path, dpi=300, bbox_inches="tight")
plt.show()

print(f"Saved to: {confusion_matrix_path}")


### ROC curve and AUC

In [ ]:
auc = roc_auc_score(y_true, y_score)
fpr, tpr, thresholds = roc_curve(y_true, y_score)

print(f"ROC AUC: {auc:.4f}")

plt.figure(figsize=(7, 6))
plt.plot(fpr, tpr, label=f"ResNet50 (AUC = {auc:.4f})")
plt.plot([0, 1], [0, 1], linestyle="--")

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve - Pneumonia Detection")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()

roc_path = OUTPUT_DIR / "roc_curve.png"
plt.savefig(roc_path, dpi=300, bbox_inches="tight")
plt.show()

print(f"Saved to: {roc_path}")


## 8. Grad-CAM and Grad-CAM++ interpretability

In [ ]:
model.eval()

target_layers = [model.layer4[-1]]

grad_cam = GradCAM(
    model=model,
    target_layers=target_layers,
)

grad_cam_pp = GradCAMPlusPlus(
    model=model,
    target_layers=target_layers,
)


def tensor_to_rgb_image(tensor):
    """Convert an ImageNet-normalized tensor [3,H,W] to RGB [H,W,3]."""

    image = tensor.detach().cpu().numpy()
    image = np.transpose(image, (1, 2, 0))

    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])

    image = image * std + mean
    image = np.clip(image, 0.0, 1.0)

    return image.astype(np.float32)


In [ ]:
gradcam_dir = OUTPUT_DIR / "gradcam"
gradcam_pp_dir = OUTPUT_DIR / "gradcam_plus_plus"

gradcam_dir.mkdir(parents=True, exist_ok=True)
gradcam_pp_dir.mkdir(parents=True, exist_ok=True)

NUM_EXAMPLES = 5

for index in range(NUM_EXAMPLES):
    image_tensor, true_label = test_dataset[index]
    input_tensor = image_tensor.unsqueeze(0).to(device)

    with torch.no_grad():
        logits = model(input_tensor)
        predicted_label = logits.argmax(dim=1).item()

    # The class with the highest model score is visualized.
    cam_map = grad_cam(
        input_tensor=input_tensor,
        targets=None,
    )[0]

    cam_pp_map = grad_cam_pp(
        input_tensor=input_tensor,
        targets=None,
    )[0]

    rgb_image = tensor_to_rgb_image(image_tensor)

    cam_overlay = show_cam_on_image(
        rgb_image,
        cam_map,
        use_rgb=True,
    )

    cam_pp_overlay = show_cam_on_image(
        rgb_image,
        cam_pp_map,
        use_rgb=True,
    )

    gradcam_path = (
        gradcam_dir
        / f"example_{index}_true_{true_label}_pred_{predicted_label}.png"
    )

    gradcam_pp_path = (
        gradcam_pp_dir
        / f"example_{index}_true_{true_label}_pred_{predicted_label}.png"
    )

    cv2.imwrite(
        str(gradcam_path),
        cv2.cvtColor(cam_overlay, cv2.COLOR_RGB2BGR),
    )

    cv2.imwrite(
        str(gradcam_pp_path),
        cv2.cvtColor(cam_pp_overlay, cv2.COLOR_RGB2BGR),
    )

    print(
        f"Example {index}: "
        f"true={class_names[true_label]} | "
        f"pred={class_names[predicted_label]}"
    )
